In [1]:
import pandas as pd

In [38]:
# ============================================================
# COMBINE COURSERA, EDX AND UDEMY DATASETS
# ============================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. DATASET FOLDER
# ============================================================

# Keep this when the notebook and datasets are in the same folder.
DATA_DIR = Path.cwd()

# Otherwise, write the full folder path:
# DATA_DIR = Path(r"C:\Users\YourName\Desktop\course_datasets")

print("Dataset folder:", DATA_DIR.resolve())


# ============================================================
# 2. STANDARD OUTPUT COLUMNS
# ============================================================

STANDARD_COLUMNS = [
    "course_name",
    "description",
    "skills",
    "subject",
    "level",
    "organization",
    "provider",
    "rating",
    "reviews_count",
    "students_enrolled",
    "lectures_count",
    "duration",
    "instructor",
    "price",
    "language",
    "image_url",
    "url",
    "certificate_type",
    "course_type",
    "source_file",
]


# ============================================================
# 3. FILE-FINDING FUNCTIONS
# ============================================================

def filename_variants(filename):
    """
    Produce filename variants by repeatedly removing extensions.

    Example:
        udemy_courses.csv.xls

    Produces:
        udemy_courses.csv.xls
        udemy_courses.csv
        udemy_courses
    """

    variants = set()
    current_name = Path(filename).name

    while True:
        variants.add(current_name.lower())

        suffix = Path(current_name).suffix

        if not suffix:
            break

        current_name = Path(current_name).stem

    return variants


def normalize_filename(filename):
    """
    Remove spaces, underscores, hyphens and punctuation for comparison.
    """

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(filename).lower(),
    )


def resolve_file(*possible_names):
    """
    Find a file while ignoring:
    - Uppercase/lowercase differences
    - Hidden extensions
    - .csv.xls endings
    - Spaces, underscores and hyphens
    """

    available_files = [
        file
        for file in DATA_DIR.iterdir()
        if file.is_file()
    ]

    # First: exact filename comparison
    for possible_name in possible_names:
        possible_lower = possible_name.lower()

        for file in available_files:
            if file.name.lower() == possible_lower:
                return file

    # Second: compare names after removing extensions
    for possible_name in possible_names:
        candidate_variants = filename_variants(
            possible_name
        )

        for file in available_files:
            file_variants = filename_variants(
                file.name
            )

            if candidate_variants.intersection(
                file_variants
            ):
                return file

    # Third: normalized filename comparison
    for possible_name in possible_names:
        candidate_variants = filename_variants(
            possible_name
        )

        candidate_normalized = {
            normalize_filename(name)
            for name in candidate_variants
        }

        for file in available_files:
            file_normalized = {
                normalize_filename(name)
                for name in filename_variants(file.name)
            }

            if candidate_normalized.intersection(
                file_normalized
            ):
                return file

    available_names = "\n".join(
        f"  - {file.name}"
        for file in available_files
    )

    raise FileNotFoundError(
        "\nCould not find any of these files:\n"
        + "\n".join(
            f"  - {name}"
            for name in possible_names
        )
        + "\n\nAvailable files are:\n"
        + available_names
    )


# ============================================================
# 4. DATASET-READING FUNCTIONS
# ============================================================

def remove_unnamed_columns(df):
    """
    Remove columns such as Unnamed: 0, Unnamed: 14, etc.
    """

    column_names = df.columns.astype(str)

    keep_columns = ~column_names.str.match(
        r"^Unnamed",
        case=False,
    )

    return df.loc[:, keep_columns].copy()


def load_table(*possible_names):
    """
    Read CSV, extensionless CSV or Excel files.

    It also supports files such as:
        udemy_courses.csv.xls

    that may contain CSV data despite the .xls extension.
    """

    path = resolve_file(*possible_names)

    errors = []

    # Try reading as CSV first
    for encoding in [
        "utf-8",
        "utf-8-sig",
        "latin-1",
        "cp1252",
    ]:
        try:
            df = pd.read_csv(
                path,
                encoding=encoding,
                low_memory=False,
            )

            df = remove_unnamed_columns(df)

            print(
                f"Loaded: {path.name:<45} "
                f"{len(df):>7,} rows"
            )

            return df, path.name

        except Exception as error:
            errors.append(
                f"CSV {encoding}: {error}"
            )

    # Try automatic delimiter detection
    for encoding in [
        "utf-8",
        "utf-8-sig",
        "latin-1",
        "cp1252",
    ]:
        try:
            df = pd.read_csv(
                path,
                encoding=encoding,
                sep=None,
                engine="python",
            )

            df = remove_unnamed_columns(df)

            print(
                f"Loaded: {path.name:<45} "
                f"{len(df):>7,} rows"
            )

            return df, path.name

        except Exception as error:
            errors.append(
                f"Automatic delimiter {encoding}: {error}"
            )

    # Try reading as an Excel workbook
    try:
        df = pd.read_excel(path)

        df = remove_unnamed_columns(df)

        print(
            f"Loaded: {path.name:<45} "
            f"{len(df):>7,} rows"
        )

        return df, path.name

    except Exception as error:
        errors.append(
            f"Excel: {error}"
        )

    raise ValueError(
        f"\nCould not read file: {path}\n\n"
        + "\n".join(errors[-5:])
    )


def load_json(*possible_names):
    """
    Read a standard JSON or JSON Lines file.
    """

    path = resolve_file(*possible_names)

    try:
        df = pd.read_json(path)

    except ValueError:
        df = pd.read_json(
            path,
            lines=True,
        )

    df = remove_unnamed_columns(df)

    print(
        f"Loaded: {path.name:<45} "
        f"{len(df):>7,} rows"
    )

    return df, path.name


# ============================================================
# 5. VALUE-CLEANING FUNCTIONS
# ============================================================

def flatten_value(value):
    """
    Convert nested lists and dictionaries into text.
    """

    if value is None:
        return None

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):
        cleaned_items = []

        for item in value:
            cleaned_item = flatten_value(item)

            if (
                cleaned_item
                and cleaned_item not in cleaned_items
            ):
                cleaned_items.append(cleaned_item)

        if not cleaned_items:
            return None

        return ", ".join(cleaned_items)

    if isinstance(value, dict):
        cleaned_items = []

        for key, item in value.items():
            cleaned_item = flatten_value(item)

            if cleaned_item:
                cleaned_items.append(
                    f"{key}: {cleaned_item}"
                )

        if not cleaned_items:
            return None

        return ", ".join(cleaned_items)

    try:
        if pd.isna(value):
            return None

    except (TypeError, ValueError):
        pass

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
        "<na>",
        "n/a",
        "not available",
    }:
        return None

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text


def coalesce(df, *column_names):
    """
    Use the first non-empty value from several columns.
    """

    result = pd.Series(
        pd.NA,
        index=df.index,
        dtype="object",
    )

    for column_name in column_names:
        if column_name not in df.columns:
            continue

        candidate = df[column_name].copy()

        candidate = candidate.map(
            lambda value: (
                pd.NA
                if flatten_value(value) is None
                else value
            )
        )

        result = result.combine_first(candidate)

    return result


def combine_text_columns(
    df,
    *column_names,
    separator=" ",
):
    """
    Combine multiple text columns into one field.
    """

    combined_values = []

    for row_index in df.index:
        parts = []

        for column_name in column_names:
            if column_name not in df.columns:
                continue

            value = flatten_value(
                df.at[row_index, column_name]
            )

            if value and value not in parts:
                parts.append(value)

        if parts:
            combined_values.append(
                separator.join(parts)
            )

        else:
            combined_values.append(pd.NA)

    return pd.Series(
        combined_values,
        index=df.index,
        dtype="object",
    )


def parse_number(value):
    """
    Convert values such as:
        4.8
        4.8 stars
        12K
        1.5M
    into numbers.
    """

    text = flatten_value(value)

    if text is None:
        return np.nan

    text = (
        text.lower()
        .replace(",", "")
    )

    match = re.search(
        r"(-?\d+(?:\.\d+)?)\s*([kmb])?",
        text,
    )

    if not match:
        return np.nan

    number = float(match.group(1))
    suffix = match.group(2)

    multipliers = {
        "k": 1_000,
        "m": 1_000_000,
        "b": 1_000_000_000,
    }

    if suffix:
        number *= multipliers[suffix]

    return number


def add_duration_unit(value, unit):
    """
    Add a unit to numerical duration values.
    """

    if value is None:
        return pd.NA

    try:
        if pd.isna(value):
            return pd.NA

    except (TypeError, ValueError):
        pass

    if isinstance(
        value,
        (int, float, np.integer, np.floating),
    ):
        number = float(value)

        if number.is_integer():
            number_text = str(int(number))
        else:
            number_text = str(number)

        return f"{number_text} {unit}"

    text = flatten_value(value)

    if text is None:
        return pd.NA

    # Do not add another unit when text already includes one
    if re.search(r"[A-Za-z]", text):
        return text

    return f"{text} {unit}"


def convert_paid_status(value):
    """
    Convert Boolean is_paid values into Paid or Free.
    """

    text = flatten_value(value)

    if text is None:
        return pd.NA

    normalized = text.lower()

    if normalized in {
        "true",
        "1",
        "yes",
    }:
        return "Paid"

    if normalized in {
        "false",
        "0",
        "no",
    }:
        return "Free"

    return text


# ============================================================
# 6. STANDARDIZATION FUNCTION
# ============================================================

def standardize_dataset(
    df,
    provider,
    source_file,
    mapping,
):
    """
    Convert one dataset into the standard column format.
    """

    output = pd.DataFrame(index=df.index)

    for column_name in STANDARD_COLUMNS:
        output[column_name] = pd.Series(
            pd.NA,
            index=df.index,
            dtype="object",
        )

    for target_column, source_columns in mapping.items():

        if callable(source_columns):
            output[target_column] = source_columns(df)

        elif isinstance(
            source_columns,
            (list, tuple),
        ):
            output[target_column] = coalesce(
                df,
                *source_columns,
            )

        else:
            output[target_column] = coalesce(
                df,
                source_columns,
            )

    # Use existing provider values when available
    if "provider" in df.columns:
        output["provider"] = coalesce(
            df,
            "provider",
        )

    output["provider"] = (
        output["provider"]
        .map(flatten_value)
        .fillna(provider)
    )

    output["source_file"] = source_file

    return output[STANDARD_COLUMNS]


# ============================================================
# 7. LOAD AND STANDARDIZE ALL DATASETS
# ============================================================

standardized_datasets = []


# ------------------------------------------------------------
# DATASET 1: Coursera
# Actual visible filename: Coursera
# ------------------------------------------------------------

df, source = load_table(
    "Coursera",
    "Coursera.csv",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Coursera",
        source_file=source,
        mapping={
            "course_name": [
                "Title",
            ],
            "subject": [
                "Subject",
            ],
            "organization": [
                "Institution",
            ],
            "course_type": [
                "Learning Product",
            ],
            "level": [
                "Level",
            ],
            "duration": [
                "Duration",
            ],
            "skills": [
                "Gained Skills",
            ],
            "rating": [
                "Rate",
            ],
            "reviews_count": [
                "Reviews",
            ],
        },
    )
)


# ------------------------------------------------------------
# DATASET 2: coursea_data
# Notice: your filename uses coursea, not coursera
# ------------------------------------------------------------

df, source = load_table(
    "coursea_data",
    "coursea_data.csv",
    "coursera_data",
    "coursera_data.csv",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Coursera",
        source_file=source,
        mapping={
            "course_name": [
                "course_title",
            ],
            "organization": [
                "course_organization",
            ],
            "certificate_type": [
                "course_Certificate_type",
            ],
            "rating": [
                "course_rating",
            ],
            "level": [
                "course_difficulty",
            ],
            "students_enrolled": [
                "course_students_enrolled",
            ],
        },
    )
)


# ------------------------------------------------------------
# DATASET 3: coursera_1000_Courses
# This file may not have an extension
# ------------------------------------------------------------

df, source = load_table(
    "coursera_1000_Courses",
    "coursera_1000_Courses.csv",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Coursera",
        source_file=source,
        mapping={
            "course_name": [
                "Course_Name",
            ],
            "organization": [
                "Company_Name",
            ],
            "skills": [
                "Skills",
            ],
            "rating": [
                "Ratings",
            ],
            "reviews_count": [
                "Reviews",
            ],
            "level": [
                "Difficulty",
            ],
            "certificate_type": [
                "Type_Of_Certificate",
            ],
            "duration": [
                "Duration",
            ],
            "image_url": [
                "Course_Banner",
            ],
        },
    )
)


# ------------------------------------------------------------
# DATASET 4: Coursera_catalog
# ------------------------------------------------------------

df, source = load_table(
    "Coursera_catalog",
    "Coursera_catalog.csv",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Coursera",
        source_file=source,
        mapping={
            "course_name": [
                "course_title",
            ],
            "url": [
                "course_URL",
            ],
            "organization": [
                "course_organization",
            ],
            "certificate_type": [
                "course_Certificate_type",
            ],
            "rating": [
                "course_rating",
            ],
            "level": [
                "course_difficulty",
            ],
            "students_enrolled": [
                "course_students_enrolled",
            ],
            "image_url": [
                "course_icon",
                "image_name",
            ],
            "skills": [
                "course_skills",
            ],
            "instructor": [
                "course_top_instructor",
            ],
        },
    )
)


# ------------------------------------------------------------
# DATASET 5: data
# This appears to contain Udemy course data
# ------------------------------------------------------------

df, source = load_table(
    "data",
    "data.csv",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Udemy",
        source_file=source,
        mapping={
            "course_name": [
                "course_name",
            ],
            "instructor": [
                "instructor",
            ],
            "url": [
                "course_url",
            ],
            "image_url": [
                "course_image",
            ],
            "description": [
                "course_description",
            ],
            "rating": [
                "reviews_avg",
            ],
            "reviews_count": [
                "reviews_count",
            ],
            "duration": [
                "course_duration",
            ],
            "lectures_count": [
                "lectures_count",
            ],
            "level": [
                "level",
            ],
            "price": [
                "price_after_discount",
                "main_price",
            ],
            "course_type": [
                "course_flag",
            ],
            "students_enrolled": [
                "students_count",
            ],
        },
    )
)


# ------------------------------------------------------------
# SHARED EDX COLUMN MAPPING
# ------------------------------------------------------------

EDX_MAPPING = {
    "course_name": [
        "title",
    ],

    "description": lambda data: combine_text_columns(
        data,
        "primary_description",
        "secondary_description",
        "tertiary_description",
        separator=" ",
    ),

    "skills": lambda data: combine_text_columns(
        data,
        "skills",
        "tags",
        separator=", ",
    ),

    "subject": [
        "subject",
    ],

    "level": [
        "level",
    ],

    "organization": [
        "partner",
        "organization_short_code_override",
    ],

    "language": [
        "language",
    ],

    "course_type": [
        "program_type",
        "product",
        "learning_type",
    ],

    "instructor": [
        "staff",
        "owners",
    ],

    "price": [
        "subscription_prices",
    ],

    "url": [
        "marketing_url",
        "external_url",
    ],

    "image_url": [
        "card_image_url",
    ],

    "students_enrolled": [
        "recent_enrollment_count",
    ],

    "duration": lambda data: coalesce(
        data,
        "weeks_to_complete",
    ).map(
        lambda value: add_duration_unit(
            value,
            "weeks",
        )
    ),
}


# ------------------------------------------------------------
# DATASET 6: edx_courses
# ------------------------------------------------------------

df, source = load_json(
    "edx_courses",
    "edx_courses.json",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="edX",
        source_file=source,
        mapping=EDX_MAPPING,
    )
)


# ------------------------------------------------------------
# DATASET 7: edx_degree_programs
# ------------------------------------------------------------

df, source = load_json(
    "edx_degree_programs",
    "edx_degree_programs.json",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="edX",
        source_file=source,
        mapping=EDX_MAPPING,
    )
)


# ------------------------------------------------------------
# DATASET 8: edx_executive_education_paidstuff
# ------------------------------------------------------------

df, source = load_json(
    "edx_executive_education_paidstuff",
    "edx_executive_education_paidstuff.json",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="edX",
        source_file=source,
        mapping=EDX_MAPPING,
    )
)


# ------------------------------------------------------------
# DATASET 9: edx_programs
# ------------------------------------------------------------

df, source = load_json(
    "edx_programs",
    "edx_programs.json",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="edX",
        source_file=source,
        mapping=EDX_MAPPING,
    )
)


# ------------------------------------------------------------
# DATASET 10: processed_coursera_data
# ------------------------------------------------------------

df, source = load_json(
    "processed_coursera_data",
    "processed_coursera_data.json",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Coursera",
        source_file=source,
        mapping={
            "course_name": [
                "course_name",
            ],
            "url": [
                "url",
            ],
            "organization": [
                "organization",
            ],
            "instructor": [
                "instructor",
            ],
            "rating": [
                "rating",
            ],
            "description": [
                "description",
            ],
            "skills": [
                "skills",
            ],
            "level": [
                "level",
            ],
            "duration": [
                "Duration",
            ],
            "reviews_count": [
                "nu_reviews",
                "reviews",
            ],
            "subject": [
                "subject",
            ],
            "course_type": [
                "type",
            ],
        },
    )
)


# ------------------------------------------------------------
# DATASET 11: webautomation_coursera
# ------------------------------------------------------------

df, source = load_table(
    "webautomation_coursera",
    "webautomation_coursera.csv",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Coursera",
        source_file=source,
        mapping={
            "course_name": [
                "title",
            ],
            "url": [
                "url",
            ],
            "organization": [
                "associated-university-institution-company",
            ],
            "course_type": [
                "type",
            ],
            "image_url": [
                "image",
            ],
            "subject": [
                "category-subject-area",
            ],
            "certificate_type": [
                "certificate-is-available",
            ],
            "description": [
                "description",
            ],
            "duration": [
                "duration",
            ],
            "language": [
                "language",
            ],
            "level": [
                "level",
            ],
            "price": [
                "price",
            ],
            "rating": [
                "rating",
            ],
        },
    )
)


# ------------------------------------------------------------
# DATASET 12: udemy_courses
#
# Your visible filename is udemy_courses.csv, but the actual
# filename may be udemy_courses.csv.xls.
# ------------------------------------------------------------

df, source = load_table(
    "udemy_courses",
    "udemy_courses.csv",
    "udemy_courses.csv.xls",
    "udemy_courses.xls",
    "udemy_courses.xlsx",
)

standardized_datasets.append(
    standardize_dataset(
        df=df,
        provider="Udemy",
        source_file=source,
        mapping={
            "course_name": [
                "course_title",
            ],
            "url": [
                "url",
            ],
            "price": [
                "price",
            ],
            "students_enrolled": [
                "num_subscribers",
            ],
            "reviews_count": [
                "num_reviews",
            ],
            "lectures_count": [
                "num_lectures",
            ],
            "level": [
                "level",
            ],
            "duration": lambda data: coalesce(
                data,
                "content_duration",
            ).map(
                lambda value: add_duration_unit(
                    value,
                    "hours",
                )
            ),
            "subject": [
                "subject",
            ],
            "course_type": lambda data: coalesce(
                data,
                "is_paid",
            ).map(convert_paid_status),
        },
    )
)


# ============================================================
# 8. COMBINE ALL DATASETS
# ============================================================

combined = pd.concat(
    standardized_datasets,
    ignore_index=True,
)

raw_row_count = len(combined)

print("\nRaw combined rows:", f"{raw_row_count:,}")


# ============================================================
# 9. CLEAN TEXT COLUMNS
# ============================================================

numeric_columns = {
    "rating",
    "reviews_count",
    "students_enrolled",
    "lectures_count",
}

for column_name in STANDARD_COLUMNS:
    if column_name not in numeric_columns:
        combined[column_name] = combined[
            column_name
        ].map(flatten_value)


# Remove rows without a valid course name
combined = combined.dropna(
    subset=["course_name"]
).copy()

combined = combined[
    combined["course_name"].str.strip() != ""
].copy()


# ============================================================
# 10. CLEAN NUMERIC COLUMNS
# ============================================================

combined["rating"] = combined[
    "rating"
].map(parse_number)

# Remove invalid ratings
combined.loc[
    ~combined["rating"].between(
        0,
        5,
        inclusive="both",
    ),
    "rating",
] = np.nan


for column_name in [
    "reviews_count",
    "students_enrolled",
    "lectures_count",
]:
    combined[column_name] = (
        combined[column_name]
        .map(parse_number)
        .round()
        .astype("Int64")
    )


# ============================================================
# 11. CREATE DUPLICATE-MATCHING KEYS
# ============================================================

def create_comparison_key(series):
    """
    Create a normalized text key for duplicate detection.
    """

    return (
        series
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(
            r"[^\w]+",
            " ",
            regex=True,
        )
        .str.strip()
    )


combined["_course_key"] = create_comparison_key(
    combined["course_name"]
)

combined["_provider_key"] = create_comparison_key(
    combined["provider"]
)

combined["_organization_key"] = create_comparison_key(
    combined["organization"]
)


# Calculate how much information each row contains
combined["_completeness"] = (
    combined[STANDARD_COLUMNS]
    .notna()
    .sum(axis=1)
)

# Put the most complete rows first
combined = combined.sort_values(
    by="_completeness",
    ascending=False,
)


# ============================================================
# 12. MERGE DUPLICATE COURSES
# ============================================================

def first_available(series):
    """
    Return the first non-empty value.
    """

    for value in series:
        if flatten_value(value) is not None:
            return value

    return pd.NA


def combine_source_files(series):
    """
    Preserve the names of every source file used for a course.
    """

    source_names = []

    for value in series:
        text = flatten_value(value)

        if text and text not in source_names:
            source_names.append(text)

    if not source_names:
        return pd.NA

    return " | ".join(source_names)


aggregation_rules = {
    column_name: first_available
    for column_name in STANDARD_COLUMNS
}

aggregation_rules["source_file"] = combine_source_files


combined = (
    combined
    .groupby(
        [
            "_course_key",
            "_provider_key",
            "_organization_key",
        ],
        as_index=False,
        sort=False,
        dropna=False,
    )
    .agg(aggregation_rules)
)


# Remove temporary columns
combined = combined.drop(
    columns=[
        "_course_key",
        "_provider_key",
        "_organization_key",
    ],
    errors="ignore",
)


# ============================================================
# 13. ADD COURSE ID
# ============================================================

combined = combined.reset_index(drop=True)

combined.insert(
    0,
    "course_id",
    range(1, len(combined) + 1),
)


# ============================================================
# 14. FINAL COLUMN ORDER
# ============================================================

FINAL_COLUMNS = [
    "course_id",
    "course_name",
    "description",
    "skills",
    "subject",
    "level",
    "organization",
    "provider",
    "rating",
    "reviews_count",
    "students_enrolled",
    "lectures_count",
    "duration",
    "instructor",
    "price",
    "language",
    "image_url",
    "url",
    "certificate_type",
    "course_type",
    "source_file",
]

combined = combined[FINAL_COLUMNS]


# ============================================================
# 15. SAVE COMPLETE DATASET
# ============================================================

complete_output_path = (
    DATA_DIR / "combined_courses.csv"
)

combined.to_csv(
    complete_output_path,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 16. CREATE 15-COLUMN APP DATASET
# ============================================================

APP_COLUMNS = [
    "course_id",
    "course_name",
    "description",
    "skills",
    "subject",
    "level",
    "organization",
    "provider",
    "rating",
    "duration",
    "instructor",
    "price",
    "language",
    "image_url",
    "url",
]

app_dataset = combined[APP_COLUMNS].copy()

app_output_path = (
    DATA_DIR / "combined_courses_app.csv"
)

app_dataset.to_csv(
    app_output_path,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 17. FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("DATASET COMBINATION COMPLETED")
print("=" * 60)

print(
    "Raw rows before duplicate removal:",
    f"{raw_row_count:,}",
)

print(
    "Final courses after duplicate removal:",
    f"{len(combined):,}",
)

print(
    "Complete dataset:",
    complete_output_path.resolve(),
)

print(
    "App dataset:",
    app_output_path.resolve(),
)

print("\nCourses by provider:")
print(
    combined["provider"]
    .value_counts(dropna=False)
)

print("\nMissing values:")
print(
    combined.isnull().sum()
)

print("\nFirst five courses:")
print(
    combined[
        [
            "course_id",
            "course_name",
            "provider",
            "organization",
            "rating",
        ]
    ].head()
)

Dataset folder: D:\Course_recomendation_Deep_Learning_Extended\dataset
Loaded: Coursera.csv                                    3,404 rows
Loaded: coursea_data.csv                                  891 rows
Loaded: coursera_1000_Courses                           1,000 rows
Loaded: Coursera_catalog.csv                              891 rows


C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)
C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)
C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. T

Loaded: data.csv                                        5,027 rows
Loaded: edx_courses.json                                1,000 rows


C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)
C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)
C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. T

Loaded: edx_degree_programs.json                           76 rows
Loaded: edx_executive_education_paidstuff.json            227 rows


...

Loaded: edx_programs.json                                 619 rows


C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)


Loaded: processed_coursera_data.json                   13,174 rows
Loaded: webautomation_coursera.csv                        242 rows
Loaded: udemy_courses.csv.xls                           3,678 rows


C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)
C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(candidate)
C:\Users\User\AppData\Local\Temp\ipykernel_25012\2008121981.py:422: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. T


Raw combined rows: 30,229

DATASET COMBINATION COMPLETED
Raw rows before duplicate removal: 30,229
Final courses after duplicate removal: 24,646
Complete dataset: D:\Course_recomendation_Deep_Learning_Extended\dataset\combined_courses.csv
App dataset: D:\Course_recomendation_Deep_Learning_Extended\dataset\combined_courses_app.csv

Courses by provider:
provider
coursera    13009
Udemy        8285
edX          1919
Coursera     1433
Name: count, dtype: int64

Missing values:
course_id                0
course_name              0
description           9498
skills               12306
subject              16199
level                   41
organization          8305
provider                 0
rating               10440
reviews_count         2268
students_enrolled    13334
lectures_count       16373
duration              1558
instructor            4404
price                16083
language             22574
image_url            21090
url                   5389
certificate_type     22842
course_t

In [39]:
df = pd.read_csv('combined_courses.csv')

In [40]:
df.shape

(24646, 21)